# 01 — Data Preparation: Skeleton Extraction & Dataset Building

**Phase 1.2** of the ActionGuard_AI project.  
This notebook extracts YOLO-Pose keypoints from every raw video clip, normalises them,  
builds a windowed dataset, applies skeleton-domain augmentation, and reports class distribution.

**Single source of truth:** All extraction/normalisation logic lives in  
`src/utils/skeleton_utils.py` — this notebook only calls those functions.

---

## Design decisions documented here

| Decision | Choice | Rationale |
|----------|--------|-----------|
| Extraction model | **YOLO-Pose `yolov8n-pose.pt`** | Already a dependency (Ultralytics in requirements.txt); COCO 17-keypoint schema matches config; faster than MediaPipe on GPU; no extra install needed. MediaPipe will be re-evaluated in Phase 2 if keypoint quality is insufficient on low-res clips. |
| Joint schema | **COCO 17-keypoint**, `(x_norm, y_norm, conf)` | Standard schema; all joints needed for distinguishing Fall vs Crouching vs Lying. Confidence kept as a third channel — zero-confidence joints are effectively masked. |
| Normalisation | **Hip-centred, torso-length-scaled** | Scale- and translation-invariant; robust to camera distance variation. Y-axis flipped to upward-positive. |
| Window size | **16 frames** (`config.WINDOW_SIZE`) — provisional | ~0.53s at 30 fps. Will be tuned in Phase 2.5 and frozen in Phase 3.1. |
| Stride (training) | **4 frames** | High overlap maximises windows from short clips (Crouching avg 1.86s = ~56 frames → ~11 windows vs 3 with stride=16). |
| Augmentation | **Rotation ±15°, Jitter σ=0.02, Time-warp 0.8–1.2×** | Applied to ALL classes equally (up to ×2 copies) so the model sees variety; no class is selectively suppressed. |


## 0 — Setup

In [ ]:
import sys, os
from pathlib import Path

# Point to project root so src/ imports work
PROJECT_ROOT = Path(os.getcwd()).parent  # notebook lives in notebooks/
sys.path.insert(0, str(PROJECT_ROOT / 'src'))

print('Project root:', PROJECT_ROOT)
print('Python:', sys.executable)

In [ ]:
import numpy as np
import json
import logging
import time
import cv2
import matplotlib.pyplot as plt

from utils.config import (
    WINDOW_SIZE, CLASS_NAMES, NUM_JOINTS, NUM_COORDS, JOINT_NAMES,
    YOLO_POSE_MODEL_PATH, LABEL_TO_ALERT_COLOR_MAPPING,
)
from utils.skeleton_utils import (
    load_pose_model,
    extract_and_normalise,
    clip_to_keypoint_sequence,
    sliding_windows,
    augment_window,
)

logging.basicConfig(level=logging.INFO, format='%(asctime)s [%(levelname)s] %(message)s', datefmt='%H:%M:%S')
log = logging.getLogger(__name__)

RAW_VIDEO_DIR = PROJECT_ROOT / 'data' / 'raw_videos'
PER_FRAME_DIR = PROJECT_ROOT / 'data' / 'skeleton_dataset' / 'per_frame'
WINDOWED_DIR  = PROJECT_ROOT / 'data' / 'skeleton_dataset' / 'windowed'
SCHEMA_PATH   = PROJECT_ROOT / 'data' / 'skeleton_dataset' / 'schema.json'

VIDEO_EXTS = {'.mp4', '.avi'}
LABEL_TO_IDX = {name: i for i, name in enumerate(CLASS_NAMES)}

print('Config loaded. WINDOW_SIZE =', WINDOW_SIZE, '| NUM_JOINTS =', NUM_JOINTS)

## 1 — Load YOLO-Pose Model

Using `yolov8n-pose.pt` (nano, fastest on CPU/GPU).  
Ultralytics will auto-download it on first run.

In [ ]:
MODEL_PATH = str(PROJECT_ROOT / YOLO_POSE_MODEL_PATH)
if not Path(MODEL_PATH).exists():
    print(f'Model not found at {MODEL_PATH} — will auto-download yolov8n-pose.pt')
    MODEL_PATH = 'yolov8n-pose.pt'

t0 = time.time()
model = load_pose_model(MODEL_PATH)
print(f'Model loaded in {time.time()-t0:.1f}s')

## 2 — Sanity Check: Single Frame

Pick one clip per class and confirm keypoints look correct before full extraction.

In [ ]:
SKELETON_EDGES = [
    (0,1),(0,2),(1,3),(2,4),          # face
    (5,6),(5,7),(7,9),(6,8),(8,10),   # arms
    (5,11),(6,12),(11,12),             # torso
    (11,13),(13,15),(12,14),(14,16),  # legs
]

fig, axes = plt.subplots(2, 4, figsize=(16, 8))
axes = axes.flatten()

for ax, cls in zip(axes, CLASS_NAMES):
    cls_dir = RAW_VIDEO_DIR / cls
    clips = sorted(p for p in cls_dir.iterdir() if p.suffix.lower() in VIDEO_EXTS)
    if not clips:
        ax.set_title(f'{cls} — no clips'); continue

    cap = cv2.VideoCapture(str(clips[0]))
    # Jump to middle of clip
    total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    cap.set(cv2.CAP_PROP_POS_FRAMES, total_frames // 2)
    ret, frame = cap.read()
    cap.release()

    if not ret:
        ax.set_title(f'{cls} — read error'); continue

    kp_norm = extract_and_normalise(frame, model)

    if kp_norm is None:
        ax.set_title(f'{cls} — NO DETECTION ⚠️')
        ax.imshow(cv2.cvtColor(frame, cv2.COLOR_BGR2RGB))
        continue

    # Draw normalised skeleton
    x, y, conf = kp_norm[:, 0], kp_norm[:, 1], kp_norm[:, 2]
    for i, (xi, yi, ci) in enumerate(zip(x, y, conf)):
        color = 'lime' if ci > 0.5 else 'orange'
        ax.plot(xi, -yi, 'o', color=color, markersize=5)  # -y so up=up
    for a, b in SKELETON_EDGES:
        if conf[a] > 0 and conf[b] > 0:
            ax.plot([x[a], x[b]], [-y[a], -y[b]], 'b-', linewidth=1.2, alpha=0.7)
    ax.set_aspect('equal')
    ax.set_title(f'{cls} ✅')
    ax.invert_yaxis() if False else None

plt.suptitle('Sanity Check — Normalised Skeletons (one clip per class)', fontsize=13)
plt.tight_layout()
plt.savefig(str(PROJECT_ROOT / 'data' / 'skeleton_dataset' / 'sanity_check_skeletons.png'), dpi=100)
plt.show()
print('Sanity check complete.')

## 3 — Full Per-Frame Extraction

Processes all 641 clips and saves `(T, 17, 3)` arrays to `data/skeleton_dataset/per_frame/`.

> **Resume-safe:** Already-extracted clips are skipped. Re-run safely after interruption.

In [ ]:
PER_FRAME_DIR.mkdir(parents=True, exist_ok=True)

per_frame_paths = {cls: [] for cls in CLASS_NAMES}
skipped = 0
t_total = time.time()

for cls in CLASS_NAMES:
    cls_vid_dir = RAW_VIDEO_DIR / cls
    cls_out_dir = PER_FRAME_DIR / cls
    cls_out_dir.mkdir(exist_ok=True)

    clips = sorted(p for p in cls_vid_dir.iterdir() if p.suffix.lower() in VIDEO_EXTS)
    print(f'\n── {cls} ({len(clips)} clips) ──')

    for vp in clips:
        out_npy = cls_out_dir / (vp.stem + '.npy')
        if out_npy.exists():
            per_frame_paths[cls].append(out_npy)
            continue  # already done

        t0 = time.time()
        seq = clip_to_keypoint_sequence(str(vp), model)

        if seq is None:
            print(f'  SKIP {vp.name} (too many missing detections)')
            skipped += 1
            continue

        np.save(str(out_npy), seq)
        per_frame_paths[cls].append(out_npy)
        print(f'  OK   {vp.name:40s}  T={seq.shape[0]:3d} frames  {time.time()-t0:.1f}s')

print(f'\nExtraction done in {(time.time()-t_total)/60:.1f} min. Skipped (unusable): {skipped}')

## 4 — Windowing

Sliding window with `stride=4` to maximise windows from short clips.

In [ ]:
STRIDE = 4

all_X, all_y = [], []
base_counts = {}

for cls in CLASS_NAMES:
    label_idx = LABEL_TO_IDX[cls]
    cls_wins = []

    for npy_path in per_frame_paths.get(cls, []):
        seq  = np.load(str(npy_path))
        wins = sliding_windows(seq, window_size=WINDOW_SIZE, stride=STRIDE)
        cls_wins.append(wins)

    if not cls_wins:
        print(f'WARNING: no windows for {cls}')
        base_counts[cls] = 0
        continue

    cls_X = np.concatenate(cls_wins, axis=0)
    base_counts[cls] = len(cls_X)
    all_X.append(cls_X)
    all_y.append(np.full(len(cls_X), label_idx, dtype=np.int64))
    print(f'{cls:<14}  base windows: {len(cls_X):>5}')

print(f'\nTotal base windows: {sum(base_counts.values())}')

## 5 — Augmentation

Apply rotation ±15°, jitter σ=0.02, time-warp 0.8–1.2×.  
Generate ×2 augmented copies per class to increase variety.

In [ ]:
AUGMENT_TIMES = 2   # augmented copies per class
RNG = np.random.default_rng(42)

for cls in CLASS_NAMES:
    if base_counts.get(cls, 0) == 0:
        continue

    label_idx = LABEL_TO_IDX[cls]
    cls_wins = []
    for npy_path in per_frame_paths.get(cls, []):
        seq  = np.load(str(npy_path))
        wins = sliding_windows(seq, window_size=WINDOW_SIZE, stride=STRIDE)
        cls_wins.append(wins)

    cls_X_base = np.concatenate(cls_wins, axis=0)
    aug_pool = []

    for _ in range(AUGMENT_TIMES):
        for w in cls_X_base:
            aug_pool.append(augment_window(w, rng=RNG))

    aug_X = np.stack(aug_pool, axis=0)
    aug_y = np.full(len(aug_X), label_idx, dtype=np.int64)
    all_X.append(aug_X)
    all_y.append(aug_y)
    print(f'{cls:<14}  +{len(aug_X):>5} augmented windows (×{AUGMENT_TIMES})')

print('Augmentation done.')

## 6 — Save Windowed Dataset

In [ ]:
X = np.concatenate(all_X, axis=0)
y = np.concatenate(all_y, axis=0)

# Shuffle
perm = RNG.permutation(len(X))
X, y = X[perm], y[perm]

WINDOWED_DIR.mkdir(parents=True, exist_ok=True)
np.save(str(WINDOWED_DIR / 'X.npy'), X)
np.save(str(WINDOWED_DIR / 'y.npy'), y)

print(f'Saved X.npy: {X.shape}  ({X.nbytes / 1e6:.1f} MB)')
print(f'Saved y.npy: {y.shape}')
print(f'dtype X={X.dtype}, y={y.dtype}')

## 7 — Class Distribution & Imbalance Report

In [ ]:
total = len(y)
counts = [(int((y == i).sum()), cls) for i, cls in enumerate(CLASS_NAMES)]

print(f'\n{"Class":<14} {"Count":>7}  {"Pct":>6}  Bar')
print('-' * 50)
for count, cls in counts:
    pct = 100.0 * count / total
    bar = '█' * int(pct / 2)
    print(f'{cls:<14} {count:>7}  {pct:>5.1f}%  {bar}')

print(f'\nTotal windows: {total}')

# Imbalance check
nonzero = [(c, n) for c, n in counts if c > 0]
max_c, max_cls = max(nonzero); min_c, min_cls = min(nonzero)
ratio = max_c / max(min_c, 1)
print(f'Imbalance ratio: {ratio:.1f}× ({max_cls} / {min_cls})')
if ratio > 3.0:
    print('⚠️  HIGH IMBALANCE — use weighted loss or oversample in training.')
else:
    print('✅ Imbalance ratio acceptable (≤ 3×).')

In [ ]:
# Bar chart
cls_labels = [n for _, n in counts]
cls_counts = [c for c, _ in counts]

colors = ['#2ecc71' if LABEL_TO_ALERT_COLOR_MAPPING.get(c) == 'GREEN'
          else '#f39c12' if LABEL_TO_ALERT_COLOR_MAPPING.get(c) == 'YELLOW'
          else '#e74c3c' for c in cls_labels]

fig, ax = plt.subplots(figsize=(10, 5))
bars = ax.bar(cls_labels, cls_counts, color=colors, edgecolor='white', linewidth=0.8)
ax.set_xlabel('Action Class')
ax.set_ylabel('Number of Windows')
ax.set_title('Class Distribution (after augmentation)\nGreen=GREEN alert  Orange=YELLOW  Red=RED')
for bar, count in zip(bars, cls_counts):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 10,
            str(count), ha='center', va='bottom', fontsize=9)
plt.tight_layout()
plt.savefig(str(PROJECT_ROOT / 'data' / 'skeleton_dataset' / 'class_distribution.png'), dpi=100)
plt.show()

## 8 — Write Schema File

In [ ]:
schema = {
    'description': '8-class action recognition skeleton dataset — ActionGuard_AI Phase 1.2',
    'extraction_model': 'yolov8n-pose.pt (COCO 17-keypoint)',
    'normalisation': {
        'origin': 'midpoint of left_hip (11) and right_hip (12)',
        'scale': 'torso length = ||hip_mid - shoulder_mid||',
        'y_axis': 'upward positive (raw pixel Y negated)',
        'fallback_scale': '1/3 of bounding-box diagonal when shoulders invisible',
    },
    'window_size': WINDOW_SIZE,
    'stride': STRIDE,
    'augment_times': AUGMENT_TIMES,
    'augmentation_params': {
        'rotation_max_deg': 15.0,
        'jitter_sigma': 0.02,
        'time_warp_range': [0.8, 1.2],
    },
    'num_joints': NUM_JOINTS,
    'num_coords': NUM_COORDS,
    'coordinate_convention': ['x_norm', 'y_norm', 'confidence'],
    'joint_names': JOINT_NAMES,
    'class_names': CLASS_NAMES,
    'label_to_index': LABEL_TO_IDX,
    'X_shape': f'(N, {WINDOW_SIZE}, {NUM_JOINTS}, {NUM_COORDS})',
    'y_shape': '(N,) — integer label index into class_names',
    'total_windows': int(len(y)),
    'class_counts': {cls: int((y == i).sum()) for i, cls in enumerate(CLASS_NAMES)},
    'files': {
        'per_frame/<class>/<clip>.npy': f'(T, {NUM_JOINTS}, {NUM_COORDS}) raw per-frame keypoints',
        'windowed/X.npy': f'(N, {WINDOW_SIZE}, {NUM_JOINTS}, {NUM_COORDS})',
        'windowed/y.npy': '(N,) integer labels',
    },
}

with open(str(SCHEMA_PATH), 'w') as f:
    json.dump(schema, f, indent=2)

print('Schema written to:', SCHEMA_PATH)
print(json.dumps(schema, indent=2))

## 9 — Quick Data Verification

In [ ]:
X_check = np.load(str(WINDOWED_DIR / 'X.npy'))
y_check = np.load(str(WINDOWED_DIR / 'y.npy'))

assert X_check.shape[1] == WINDOW_SIZE,  f'Expected W={WINDOW_SIZE}, got {X_check.shape[1]}'
assert X_check.shape[2] == NUM_JOINTS,   f'Expected J={NUM_JOINTS}, got {X_check.shape[2]}'
assert X_check.shape[3] == NUM_COORDS,   f'Expected C={NUM_COORDS}, got {X_check.shape[3]}'
assert len(X_check) == len(y_check),     'X and y length mismatch'
assert set(np.unique(y_check)).issubset(set(range(len(CLASS_NAMES)))), 'Unexpected label value'
assert X_check.dtype == np.float32,       f'Expected float32, got {X_check.dtype}'

print('✅ All assertions passed.')
print(f'   X shape : {X_check.shape}')
print(f'   y shape : {y_check.shape}')
print(f'   X range : [{X_check.min():.3f}, {X_check.max():.3f}]')
print(f'   y unique: {sorted(np.unique(y_check).tolist())}')

---
## ✅ Phase 1.2 Complete

**Outputs:**
- `data/skeleton_dataset/per_frame/<class>/<clip>.npy` — raw per-frame keypoints
- `data/skeleton_dataset/windowed/X.npy` — windowed + augmented dataset
- `data/skeleton_dataset/windowed/y.npy` — integer labels
- `data/skeleton_dataset/schema.json` — full dataset documentation
- `data/skeleton_dataset/class_distribution.png` — distribution chart
- `data/skeleton_dataset/sanity_check_skeletons.png` — visual QC

**Next step:** Phase 2 — integrate YOLO-Pose into `src/pipeline/detector.py` and confirm  
its output matches what this notebook produces (same joint order, same normalisation).